# Snapshot comparison

This notebook compares a **candidate** BCSD run against a stored **snapshot**
baseline, leaf by leaf, using the same `srm.snapshot.compare` engine as the
`pytest -m snapshot` gate and the `bcsd compare` CLI. Its PASS/FAIL verdict is
therefore identical to the gate's — the notebook adds visual diagnostics
(difference maps, drift heatmap, time series, distributions) on top of the same
tolerance-aware comparison.

It is parameterized with [papermill](https://papermill.readthedocs.io/): the
first code cell is tagged `parameters`, so `snapshot_uri`, `candidate_uri`,
`branch`, `scenarios`, `variables`, and `time_index` can be overridden at
execution time.

In [ ]:
snapshot_uri = "s3://carbonplan-srm/output/production/CESM2-WACCM-ERA5-global.icechunk"
candidate_uri = (
    "s3://carbonplan-scratch/srm/output/qa/CESM2-WACCM-ERA5-lat-35to-22_lon16to33.icechunk"
)
branch = "main"
scenarios = None  # None = all known groups; or e.g. ["g6_1p5k"]
variables = None  # None = all; or e.g. ["tas", "pr"]
time_index = 0
# Note: members differ by scenario (historical/g6_1p5k = 003, ssp245 = 008);
# leaf_da() below discovers the member per node, so none is hardcoded.

In [ ]:
import matplotlib.pyplot as plt
import xarray as xr

from srm.config import SCENARIO_TO_GROUP
from srm.snapshot.compare import compare
from srm.validation import _open_output_datatree

snapshot_tree = _open_output_datatree(snapshot_uri, branch=branch)
candidate_tree = _open_output_datatree(candidate_uri, branch=branch)
KNOWN_GROUPS = set(SCENARIO_TO_GROUP.values())  # exclude e.g. debiased_coarse
DIFFS = {}


def leaf_da(tree, scenario, variable):
    """Return the member-level DataArray for one (scenario, variable) leaf.

    The member is discovered as the single child node (historical/g6_1p5k use
    003, ssp245 uses 008), so no member ID is hardcoded.
    """
    node = tree[f"{scenario}/{variable}"]
    member = next(iter(node.children))
    return node[member].to_dataset(inherit=False)[variable]

In [ ]:
def compare_leaf(scenario, variable):
    s = leaf_da(snapshot_tree, scenario, variable)
    c = leaf_da(candidate_tree, scenario, variable)
    report = compare(c.to_dataset(name=variable), s.to_dataset(name=variable))
    leaf = report.leaves[0]
    status = "PASS" if leaf.within_tol else "FAIL"
    print(
        f"{scenario}/{variable}: {status}  max_abs={leaf.max_abs_diff:.3e}  "
        f"rmse={leaf.rmse:.3e}  frac>tol={leaf.frac_over_tol:.3e}"
    )
    s0 = s.isel(time=time_index)
    c0 = c.isel(time=time_index)
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    s0.plot(ax=axes[0])
    axes[0].set_title("snapshot")
    c0.plot(ax=axes[1])
    axes[1].set_title("candidate")
    (c0 - s0).plot(ax=axes[2], cmap="bwr", center=0)
    axes[2].set_title("difference")
    plt.tight_layout()
    plt.show()
    DIFFS[(scenario, variable)] = leaf
    return leaf

In [ ]:
scen_list = scenarios or sorted(
    KNOWN_GROUPS & set(snapshot_tree.children) & set(candidate_tree.children)
)
for scenario in scen_list:
    snap_vars = set(snapshot_tree[scenario].children)
    cand_vars = set(candidate_tree[scenario].children)
    for variable in sorted(snap_vars & cand_vars):
        if variables and variable not in variables:
            continue
        compare_leaf(scenario, variable)

In [ ]:
import numpy as np
import pandas as pd

rows = sorted({v for _, v in DIFFS})
cols = sorted({s for s, _ in DIFFS})
grid = pd.DataFrame(np.nan, index=rows, columns=cols)
for (s, v), leaf in DIFFS.items():
    grid.loc[v, s] = leaf.frac_over_tol
fig, ax = plt.subplots(figsize=(1.5 * len(cols) + 2, 0.5 * len(rows) + 2))
im = ax.imshow(grid.values, aspect="auto", cmap="Reds")
ax.set_xticks(range(len(cols)))
ax.set_xticklabels(cols, rotation=45, ha="right")
ax.set_yticks(range(len(rows)))
ax.set_yticklabels(rows)
ax.set_title("fraction of cells over tolerance")
fig.colorbar(im, ax=ax)
plt.show()

In [ ]:
scenario = scen_list[0]
vars_here = sorted(set(snapshot_tree[scenario].children) & set(candidate_tree[scenario].children))
diffs = []
for v in vars_here:
    s = leaf_da(snapshot_tree, scenario, v).isel(time=time_index)
    c = leaf_da(candidate_tree, scenario, v).isel(time=time_index)
    diffs.append((c - s).expand_dims(variable=[v]))
diff_da = xr.concat(diffs, dim="variable")
diff_da.plot(col="variable", col_wrap=4, cmap="bwr", center=0, robust=True)
plt.show()

In [ ]:
for v in vars_here:
    s = leaf_da(snapshot_tree, scenario, v).mean(["lat", "lon"])
    c = leaf_da(candidate_tree, scenario, v).mean(["lat", "lon"])
    fig, ax = plt.subplots(figsize=(10, 3))
    s.plot(ax=ax, label="snapshot")
    c.plot(ax=ax, label="candidate")
    ax.set_title(f"{scenario}/{v} domain mean")
    ax.legend()
    plt.show()

In [ ]:
for v in vars_here:
    s = leaf_da(snapshot_tree, scenario, v)
    c = leaf_da(candidate_tree, scenario, v)
    fig, ax = plt.subplots(figsize=(8, 3))
    s.plot.hist(bins=50, histtype="step", density=True, ax=ax, label="snapshot")
    c.plot.hist(bins=50, histtype="step", density=True, ax=ax, label="candidate")
    ax.set_title(f"{scenario}/{v} distribution")
    ax.legend()
    plt.show()

In [ ]:
all_pass = all(leaf.within_tol for leaf in DIFFS.values())
print("OVERALL:", "PASS (no change beyond tolerance)" if all_pass else "FAIL (changes detected)")
for (s, v), leaf in sorted(DIFFS.items()):
    if not leaf.within_tol:
        print(
            f"  changed: {s}/{v}  max_abs={leaf.max_abs_diff:.3e}  frac>tol={leaf.frac_over_tol:.3e}"
        )